# 03 · RAG evaluation: retrieval + grounded composition

End-to-end check of Project 1 against a small fixture index built in notebook memory — same composer and FAISS store code as production.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('../../services/model-server'))
import numpy as np
from app.faiss_store import FaissStore
from app.answer_composer import AnswerComposer
from app.embed import HashEmbedder

emb = HashEmbedder(dim=64)
store = FaissStore(dim=64)
rows = [json.loads(l) for l in open('../../ml/data/generated/train.jsonl')][:120]

In [ ]:
# index descriptions as chunk evidence
entries = []
for i, r in enumerate(rows):
    vec = emb.encode([r['description']])[0]
    entries.append(dict(faiss_id=i, item_id=r['item_id'], modality='text',
                        text=r['description'], vector=vec.tolist()))
store.upsert(entries)
print('indexed:', store.counts())

In [ ]:
# retrieval: ask about a known feature
q = f"Which item features {rows[0]['spec']['features'][0]}?"
hits = store.search(text_vector=emb.encode([q])[0], top_k=6)
print('question:', q)
print('top-1 item:', hits[0]['item_id'], 'gold:', rows[0]['item_id'], 'score:', round(hits[0]['score'], 3))

In [ ]:
# grounded composition + citation gate
composer = AnswerComposer(0.35)
out = composer.compose(q, hits, mode='extractive')
print('answer:', out['answer'])
print('passed:', out['citation_check']['passed'])
print('citations:', [(s['index'], s['citations']) for s in out['sentences']])

In [ ]:
# the gate: an ungrounded question must refuse to answer
bad = composer.compose('What is the warranty on the lunar rover?', hits)
print('answer:', bad['answer'])
assert bad['citation_check']['passed'] is False
print('citation gate held ✔')

In [ ]:
# hit@k / MRR over the generated qrels
qrels = [json.loads(l) for l in open('../../ml/data/generated/qrels.jsonl')][:50]
hit1 = rr = 0
for q in qrels:
    hs = store.search(text_vector=emb.encode([q['question']])[0], top_k=6)
    rank = next((i + 1 for i, h in enumerate(hs) if h['item_id'] == q['item_id']), None)
    hit1 += bool(rank and rank <= 1); rr += 1 / rank if rank else 0
print(json.dumps({'hit@1': hit1 / len(qrels), 'mrr': rr / len(qrels)}, indent=2))